In [1]:
import requests
import json
import pandas as pd
import joblib
from datetime import datetime

print("Imports OK")

Imports OK


In [2]:
import sys
sys.path.append('../notebooks')

# Charger le modèle v2
rf_model_v2 = joblib.load('../models/rf_phishing_model_v2.pkl')
print("Modèle chargé !")

Modèle chargé !


In [3]:
from urllib.parse import urlparse

def extract_features_v2(url):
    try:
        if not url.startswith('http'):
            url = 'http://' + url
        
        parsed = urlparse(url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''
    
    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.free.fr', '.xyz', '.top']
    suspicious_words = ['login', 'secure', 'update', 'verify', 'account', 
                       'banking', 'paypal', 'amazon', 'apple', 'microsoft']
    
    return {
        'length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'num_slashes': url.count('/'),
        'num_digits': sum(c.isdigit() for c in url),
        'num_at': url.count('@'),
        'num_percent': url.count('%'),
        'num_question': url.count('?'),
        'num_equal': url.count('='),
        'num_underscore': url.count('_'),
        'has_https': int(url.startswith('https')),
        'has_ip': int(any(part.isdigit() for part in domain.split('.'))),
        'domain_length': len(domain),
        'path_length': len(path),
        'has_suspicious_tld': int(any(url.endswith(tld) or tld+'/' in url for tld in suspicious_tlds)),
        'has_suspicious_word': int(any(word in domain.lower() for word in suspicious_words)),
        'num_subdomains': len(domain.split('.')) - 2 if len(domain.split('.')) > 2 else 0,
        'has_at_in_url': int('@' in url),
    }

print("Fonction extract_features_v2 définie !")

Fonction extract_features_v2 définie !


In [6]:
def fetch_urlhaus_urls(limit=100):
    print("Téléchargement du feed URLhaus...")
    
    # Export CSV public - pas besoin d'authentification
    response = requests.get(
        "https://urlhaus.abuse.ch/downloads/csv_recent/",
        timeout=30
    )
    
    lines = response.text.split('\n')
    
    urls = []
    raw_data = []
    
    for line in lines:
        # Ignorer les commentaires et lignes vides
        if line.startswith('#') or line.strip() == '':
            continue
        
        parts = line.split(',')
        if len(parts) >= 3:
            url = parts[2].strip().strip('"')
            if url.startswith('http'):
                urls.append(url)
                raw_data.append({'url': url, 'status': parts[3].strip().strip('"') if len(parts) > 3 else 'unknown'})
        
        if len(urls) >= limit:
            break
    
    print(f"{len(urls)} URLs récupérées depuis URLhaus")
    return urls, raw_data

urls, raw_data = fetch_urlhaus_urls(limit=100)
print(f"\nExemple d'URLs récupérées :")
for url in urls[:5]:
    print(f"  {url}")

Téléchargement du feed URLhaus...
100 URLs récupérées depuis URLhaus

Exemple d'URLs récupérées :
  http://117.9.3.219:50790/i
  http://115.46.158.248:38896/i
  http://211.73.163.154:36543/.i
  http://78.171.62.85:50435/bin.sh
  http://123.5.170.80:52707/i


In [9]:
import warnings
warnings.filterwarnings('ignore')

def analyze_urls(urls, raw_data):
    print("Analyse en cours...")
    
    # Extraire toutes les features en une fois
    all_features = [extract_features_v2(url) for url in urls]
    features_df = pd.DataFrame(all_features)
    
    # Prédiction batch — une seule fois pour toutes les URLs
    predictions = rf_model_v2.predict(features_df)
    probabilities = rf_model_v2.predict_proba(features_df)
    confidences = [round(max(prob) * 100, 2) for prob in probabilities]
    
    results = []
    for i, url in enumerate(urls):
        results.append({
            'url': url,
            'prediction': 'MALICIOUS' if predictions[i] == 1 else 'BENIGN',
            'confidence': confidences[i],
            'status': raw_data[i].get('status', 'unknown'),
            'timestamp': datetime.now().isoformat()
        })
    
    df_results = pd.DataFrame(results)
    
    print(f"Total analysées     : {len(df_results)}")
    print(f"Détectées MALICIOUS : {len(df_results[df_results['prediction'] == 'MALICIOUS'])}")
    print(f"Détectées BENIGN    : {len(df_results[df_results['prediction'] == 'BENIGN'])}")
    
    return df_results

df_results = analyze_urls(urls, raw_data)
print("\nAperçu :")
print(df_results.head(10))

Analyse en cours...
Total analysées     : 100
Détectées MALICIOUS : 91
Détectées BENIGN    : 9

Aperçu :
                                                 url prediction  confidence  \
0                         http://117.9.3.219:50790/i  MALICIOUS       100.0   
1                      http://115.46.158.248:38896/i  MALICIOUS       100.0   
2                     http://211.73.163.154:36543/.i  MALICIOUS       100.0   
3                   http://78.171.62.85:50435/bin.sh  MALICIOUS       100.0   
4                        http://123.5.170.80:52707/i  MALICIOUS       100.0   
5                 http://119.117.87.164:59176/bin.sh  MALICIOUS       100.0   
6                      http://119.117.87.164:59176/i  MALICIOUS       100.0   
7  https://kyqwfrjr.betturkey.bet/6c3d1924-bdc3-4...  MALICIOUS        91.0   
8  https://rddj.jadoou.boats/d0dfe87c-66d1-4a9b-b...  MALICIOUS        96.0   
9  https://nsuoqcej.betturkey.bet/1b96bba8-7963-4...  MALICIOUS        89.0   

    status               

In [10]:
def generate_alert_report(df_results):
    malicious = df_results[df_results['prediction'] == 'MALICIOUS']
    
    report = {
        'report_id': f"CMRPI-{datetime.now().strftime('%Y%m%d-%H%M%S')}",
        'generated_at': datetime.now().isoformat(),
        'source': 'URLhaus - Abuse.ch',
        'summary': {
            'total_analyzed': len(df_results),
            'malicious_detected': len(malicious),
            'benign': len(df_results) - len(malicious),
            'detection_rate': f"{len(malicious)/len(df_results)*100:.1f}%"
        },
        'alerts': malicious[['url', 'confidence', 'status', 'timestamp']].to_dict(orient='records')
    }
    
    # Sauvegarder le rapport
    report_path = f"../data/alert_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"Rapport généré : {report_path}")
    print(f"\n=== RÉSUMÉ ===")
    print(f"Report ID       : {report['report_id']}")
    print(f"Source          : {report['source']}")
    print(f"Total analysées : {report['summary']['total_analyzed']}")
    print(f"MALICIOUS       : {report['summary']['malicious_detected']}")
    print(f"Taux détection  : {report['summary']['detection_rate']}")
    
    return report

report = generate_alert_report(df_results)

Rapport généré : ../data/alert_report_20260712_160654.json

=== RÉSUMÉ ===
Report ID       : CMRPI-20260712-160654
Source          : URLhaus - Abuse.ch
Total analysées : 100
MALICIOUS       : 91
Taux détection  : 91.0%
